In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [2]:
# Copilot helped generate the basic operations and autograd examples for PyTorch tensors.

# From a Python list
t_list = torch.tensor([1.0, 2.0, 3.0, 4.0])
print("Tensor from list:", t_list)

# From a nested list (2-D)
t_2d = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
print("2-D tensor:\n", t_2d)

# Random normal
t_rand = torch.randn(3, 4)
print("Random normal (3×4):\n", t_rand)

# Zeros and ones
t_zeros = torch.zeros(2, 3)
t_ones  = torch.ones(2, 3)
print("Zeros:\n", t_zeros)
print("Ones:\n",  t_ones)

a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

# Element-wise addition
print("\na + b =\n", a + b)

# Element-wise multiplication
print("a * b =\n", a * b)

# Matrix multiplication
print("a @ b =\n", a @ b)

# Scalar operations
print("a * 2 =\n", a * 2)
print("a.sum() =", a.sum().item())
print("a.mean() =", a.mean().item())

# Simple scalar example: y = x² + 2x + 1  →  dy/dx = 2x + 2
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1
y.backward()
print(f"\nAutograd (scalar):")
print(f"  x = {x.item()}, y = x²+2x+1 = {y.item()}")
print(f"  dy/dx = {x.grad.item()}")   # expected: 8.0

# Vector example: z = sum(w²)  →  dz/dw = 2w
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
z = (w ** 2).sum()
z.backward()
print(f"\nAutograd (vector):")
print(f"  w      = {w.tolist()}")
print(f"  z      = sum(w²) = {z.item()}")
print(f"  dz/dw  = {w.grad.tolist()}")   # expected: [2, 4, 6]


Tensor from list: tensor([1., 2., 3., 4.])
2-D tensor:
 tensor([[1., 2., 3.],
        [4., 5., 6.]])
Random normal (3×4):
 tensor([[-1.3977, -0.2193,  0.0104,  0.5640],
        [ 0.9059,  0.1874, -1.3160,  0.3132],
        [ 0.4570,  1.0012,  0.3672,  0.7886]])
Zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
Ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])

a + b =
 tensor([[ 6.,  8.],
        [10., 12.]])
a * b =
 tensor([[ 5., 12.],
        [21., 32.]])
a @ b =
 tensor([[19., 22.],
        [43., 50.]])
a * 2 =
 tensor([[2., 4.],
        [6., 8.]])
a.sum() = 10.0
a.mean() = 2.5

Autograd (scalar):
  x = 3.0, y = x²+2x+1 = 16.0
  dy/dx = 8.0

Autograd (vector):
  w      = [1.0, 2.0, 3.0]
  z      = sum(w²) = 14.0
  dz/dw  = [2.0, 4.0, 6.0]


In [ ]:
# Copilot Generated all of the training and model evaluation code for the Iris dataset

# ── Load & preprocess Iris dataset ──────────────────────────────────────────
iris = load_iris()
X, y = iris.data.astype(np.float32), iris.target  # (150, 4), (150,)

# 80/20 train-test split (stratified so all 3 classes are represented)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardise features (fit on train only)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

# Convert to tensors
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train).long()
X_test_t  = torch.from_numpy(X_test)
y_test_t  = torch.from_numpy(y_test).long()

# DataLoader for batching
train_ds     = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

print(f"Train samples: {len(X_train_t)}, Test samples: {len(X_test_t)}")
print(f"Input features: {X_train_t.shape[1]}, Classes: {len(iris.target_names)}")


Train samples: 120, Test samples: 30
Input features: 4, Classes: 3


In [6]:
# ── Model definition ────────────────────────────────────────────────────────
class SimpleClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        return x  # raw logits — CrossEntropyLoss applies softmax internally

INPUT_SIZE  = X_train_t.shape[1]   # 4
HIDDEN_SIZE = 16
NUM_CLASSES = len(iris.target_names)  # 3

model = SimpleClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES)
print(model)


SimpleClassifier(
  (layer1): Linear(in_features=4, out_features=16, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=16, out_features=3, bias=True)
)


In [7]:
# ── Training loop ────────────────────────────────────────────────────────────
EPOCHS    = 100
LR        = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(X_batch)

    if epoch % 10 == 0:
        avg_loss = running_loss / len(train_ds)
        print(f"Epoch {epoch:3d}/{EPOCHS}  |  Loss: {avg_loss:.4f}")


Epoch  10/100  |  Loss: 0.8360
Epoch  20/100  |  Loss: 0.5792
Epoch  30/100  |  Loss: 0.4329
Epoch  40/100  |  Loss: 0.3396
Epoch  50/100  |  Loss: 0.2733
Epoch  60/100  |  Loss: 0.2270
Epoch  70/100  |  Loss: 0.1920
Epoch  80/100  |  Loss: 0.1645
Epoch  90/100  |  Loss: 0.1434
Epoch 100/100  |  Loss: 0.1271


In [9]:
# ── Evaluate on test set & save model ───────────────────────────────────────
model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    predictions = test_logits.argmax(dim=1)
    correct     = (predictions == y_test_t).sum().item()
    accuracy    = correct / len(y_test_t) * 100

print(f"\nTest accuracy: {correct}/{len(y_test_t)} = {accuracy:.1f}%")

# Quick per-class breakdown
for class_idx, class_name in enumerate(iris.target_names):
    mask      = y_test_t == class_idx
    class_acc = (predictions[mask] == y_test_t[mask]).sum().item() / mask.sum().item() * 100
    print(f"  {class_name}: {class_acc:.0f}%")

# Save model weights and scaler so main.py can load them directly
import joblib
MODEL_PATH  = "model.pth"
SCALER_PATH = "scaler.pkl"
torch.save(model.state_dict(), MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)
print(f"\nModel saved  → {MODEL_PATH}")
print(f"Scaler saved → {SCALER_PATH}")



Test accuracy: 29/30 = 96.7%
  setosa: 100%
  versicolor: 90%
  virginica: 100%

Model saved  → model.pth
Scaler saved → scaler.pkl
